# VERA — CALVIN Ablation Training
**9 ablations × 3 seeds on CALVIN D→D**

### Before running:
1. `Runtime → Change runtime type → T4 GPU` (free) or A100 (Pro)
2. Run cells top to bottom — dataset downloads automatically inside Colab (first time only, saves to Drive)

### If session dies mid-run:
- Checkpoints are saved to Drive after every epoch — nothing is lost
- Re-run all setup cells (dataset skips re-download if already on Drive)
- In Cell 7, set `START_FROM` to the ablation index that was running when it died

In [1]:
# ── Cell 1: Check GPU ────────────────────────────────────────────────────────
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    print('VRAM:  ', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('WARNING: No GPU found. Go to Runtime → Change runtime type → GPU')

GPU available: True
Device: Tesla T4
VRAM:   15.6 GB


In [2]:
# ── Cell 2: Mount Google Drive ───────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# All checkpoints and results will be saved here — survives session crashes
DRIVE_ROOT = '/content/drive/MyDrive/VERA_CALVIN'

import os
os.makedirs(DRIVE_ROOT, exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/checkpoints', exist_ok=True)
print('Drive mounted. Output dir:', DRIVE_ROOT)

Mounted at /content/drive
Drive mounted. Output dir: /content/drive/MyDrive/VERA_CALVIN


In [3]:
# ── Cell 3: Install dependencies ─────────────────────────────────────────────
# torch / numpy / PIL already in Colab — only need CLIP and ftfy
!pip install -q git+https://github.com/openai/CLIP.git
!pip install -q ftfy regex

import clip, numpy, PIL, yaml
print('clip:', clip.__version__ if hasattr(clip,'__version__') else 'ok')
print('torch:', torch.__version__)
print('All dependencies ready')

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.3 MB/s eta 0:00:00
clip: ok
torch: 2.10.0+cu128
All dependencies ready


In [4]:
# ── Cell 4: Clone repo ───────────────────────────────────────────────────────
import os

REPO_URL  = 'https://github.com/sara-kaz/RLConditionedVLA.git'
REPO_DIR  = '/content/RLConditionedVLA'

if os.path.exists(REPO_DIR):
    print('Repo already cloned — pulling latest changes')
    !cd {REPO_DIR} && git pull
else:
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print('Working dir:', os.getcwd())
!ls

Cloning into '/content/RLConditionedVLA'...
remote: Enumerating objects: 175, done.
remote: Counting objects: 100% (28/28), done.
remote: Compressing objects: 100% (26/26), done.
remote: Total 175 (delta 12), reused 8 (delta 2), pack-reused 147 (from 1)
Receiving objects: 100% (175/175), 41.99 MiB | 22.27 MiB/s, done.
Resolving deltas: 100% (61/61), done.
Working dir: /content/RLConditionedVLA
configs  envs	     README.md	     requirements.txt  VERA_CALVIN_Colab.ipynb
data	 evaluation  references.bib  scripts
docs	 models      REPORT.md	     training


In [6]:
# ── Optional: Clear Drive cache (run only if you want to re-download everything) ─
# Normally skip this cell.
# Run it only if the cache is corrupt or you want to force a fresh download.

CLEAR_CACHE = False   # ← change to True then run this cell to wipe cache

if CLEAR_CACHE:
    import shutil, os
    tar   = f'{DRIVE_ROOT}/calvin_data.tar'
    cdir  = f'{DRIVE_ROOT}/calvin_cdir.json'
    local = '/content/calvin_data'
    if os.path.exists(tar):   os.remove(tar);      print(f'Removed {tar}')
    if os.path.exists(cdir):  os.remove(cdir);     print(f'Removed {cdir}')
    if os.path.exists(local): shutil.rmtree(local); print(f'Removed {local}')
    print('Cache cleared — re-run Cell 5 to re-download')
else:
    print('Cache intact (set CLEAR_CACHE = True to wipe)')


In [8]:
# ── Cell 5: CALVIN dataset via ZIP64 batch range requests + Drive tar cache ─────
#
# Server returns 404 for individual files — only the 177 GB ZIP exists.
# Strategy:
#   First session: parse ZIP64 central directory (cached to Drive as JSON),
#     batch-download episode NPZs in 80 MB chunks (~647 requests, ~40 min),
#     then pack everything into a single tar and copy to Drive (~5 min).
#   Later sessions: extract tar from Drive (~3 min). No re-download needed.
#
import os, struct, json as _json, shutil, urllib.request, tarfile, zlib
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm

CALVIN_ZIP_URL = 'http://calvin.cs.uni-freiburg.de/dataset/task_D_D.zip'
CALVIN_PATH    = '/content/calvin_data/task_D_D'
CDIR_CACHE     = f'{DRIVE_ROOT}/calvin_cdir.json'
TAR_DRIVE      = f'{DRIVE_ROOT}/calvin_data.tar'
TAR_LOCAL      = '/content/calvin_data_tmp.tar'   # staging on local SSD

TRAIN_EP = 5124   # all annotated training episodes (~55 K frames)
VAL_EP   = 1011   # all annotated validation episodes (~11 K frames)

# Create local directory structure
for split in ['training', 'validation']:
    os.makedirs(f'{CALVIN_PATH}/{split}/lang_annotations', exist_ok=True)

# ── HTTP range helper ─────────────────────────────────────────────────────────
def range_get(url, start, end, timeout=120):
    req = urllib.request.Request(url)
    req.add_header('Range', f'bytes={start}-{end}')
    resp = urllib.request.urlopen(req, timeout=timeout)
    chunks = []
    while True:
        c = resp.read(4 * 1024 * 1024)
        if not c:
            break
        chunks.append(c)
    return b''.join(chunks)

# ── ZIP64 central directory parser (cached to Drive JSON, built once) ─────────
_file_index = None

def get_file_index():
    global _file_index
    if _file_index is not None:
        return _file_index
    if os.path.exists(CDIR_CACHE):
        print('Loading ZIP index from Drive...')
        with open(CDIR_CACHE) as f:
            _file_index = _json.load(f)
        print(f'  {len(_file_index):,} files indexed')
        return _file_index
    print('Building ZIP64 index (downloads ~73 MB once)...')
    resp = urllib.request.urlopen(
        urllib.request.Request(CALVIN_ZIP_URL, method='HEAD'), timeout=15)
    zip_size = int(resp.headers['Content-Length'])
    print(f'  ZIP size: {zip_size/1e9:.1f} GB')
    tail = range_get(CALVIN_ZIP_URL, zip_size - 65536, zip_size - 1)
    li   = tail.rfind(b'\x50\x4b\x06\x07')
    assert li >= 0, 'ZIP64 EOCD locator not found'
    eocd64_off = struct.unpack_from('<Q', tail, li + 8)[0]
    eocd64 = range_get(CALVIN_ZIP_URL, eocd64_off, eocd64_off + 55)
    assert eocd64[:4] == b'\x50\x4b\x06\x06'
    cdir_size   = struct.unpack_from('<Q', eocd64, 40)[0]
    cdir_offset = struct.unpack_from('<Q', eocd64, 48)[0]
    print(f'  Central dir: {cdir_size/1e6:.1f} MB at offset {cdir_offset:,}')
    cdir = range_get(CALVIN_ZIP_URL, cdir_offset, cdir_offset + cdir_size - 1, timeout=300)
    _file_index = {}
    pos = 0
    while pos + 46 <= len(cdir) and cdir[pos:pos+4] == b'\x50\x4b\x01\x02':
        comp_sz   = struct.unpack_from('<I', cdir, pos + 20)[0]
        uncomp_sz = struct.unpack_from('<I', cdir, pos + 24)[0]
        name_len  = struct.unpack_from('<H', cdir, pos + 28)[0]
        extra_len = struct.unpack_from('<H', cdir, pos + 30)[0]
        cmnt_len  = struct.unpack_from('<H', cdir, pos + 32)[0]
        lhdr_off  = struct.unpack_from('<I', cdir, pos + 42)[0]
        name  = cdir[pos+46 : pos+46+name_len].decode('utf-8', errors='replace')
        extra = cdir[pos+46+name_len : pos+46+name_len+extra_len]
        ep = 0
        while ep + 4 <= len(extra):
            ex_id = struct.unpack_from('<H', extra, ep)[0]
            ex_sz = struct.unpack_from('<H', extra, ep + 2)[0]
            if ex_id == 0x0001:
                vp = ep + 4
                if uncomp_sz == 0xFFFFFFFF and vp + 8 <= ep + 4 + ex_sz:
                    uncomp_sz = struct.unpack_from('<Q', extra, vp)[0]; vp += 8
                if comp_sz == 0xFFFFFFFF and vp + 8 <= ep + 4 + ex_sz:
                    comp_sz   = struct.unpack_from('<Q', extra, vp)[0]; vp += 8
                if lhdr_off == 0xFFFFFFFF and vp + 8 <= ep + 4 + ex_sz:
                    lhdr_off  = struct.unpack_from('<Q', extra, vp)[0]
                break
            ep += 4 + ex_sz
        if name.startswith('task_D_D/'):
            name = name[len('task_D_D/'):]
        if name and not name.endswith('/'):
            _file_index[name] = [lhdr_off, comp_sz]
        pos += 46 + name_len + extra_len + cmnt_len
    print(f'  Parsed {len(_file_index):,} entries')
    with open(CDIR_CACHE, 'w') as f:
        _json.dump(_file_index, f)
    print(f'  Saved to Drive: {CDIR_CACHE}')
    return _file_index

# ── Download a single file from ZIP ──────────────────────────────────────────
def download_one(key):
    idx = get_file_index()
    lhdr_off, comp_sz = idx[key]
    lhdr = range_get(CALVIN_ZIP_URL, lhdr_off, lhdr_off + 29, timeout=15)
    assert lhdr[:4] == b'\x50\x4b\x03\x04'
    method    = struct.unpack_from('<H', lhdr,  8)[0]
    name_len  = struct.unpack_from('<H', lhdr, 26)[0]
    extra_len = struct.unpack_from('<H', lhdr, 28)[0]
    data_off  = lhdr_off + 30 + name_len + extra_len
    raw = range_get(CALVIN_ZIP_URL, data_off, data_off + comp_sz - 1, timeout=90)
    return zlib.decompress(raw, -15) if method == 8 else raw

# ── Batch range-request download (~80 MB per HTTP request) ────────────────────
# Groups consecutive ZIP entries into large chunks — ~647 requests total
# instead of 55 000 individual requests (40 min vs 17 hours).
def batch_download(split, frame_ids, local_dir):
    idx = get_file_index()
    file_info = []
    for i in frame_ids:
        key = f'{split}/episode_{i:07d}.npz'
        if key in idx:
            lo, cs = idx[key]
            file_info.append((i, lo, cs))
    if not file_info:
        print('  Nothing to download')
        return
    file_info.sort(key=lambda x: x[1])   # sort by offset in ZIP
    CHUNK = 80 * 1024 * 1024   # 80 MB per request
    GAP   =  2 * 1024 * 1024   # merge consecutive if gap < 2 MB
    chunks, curr, c0, c1 = [], [], None, None
    for i, lo, cs in file_info:
        fe = lo + 512 + cs   # estimated file end (512 = local header overhead)
        if c0 is None:
            c0, c1, curr = lo, fe, [(i, lo, cs)]
        elif fe - c0 <= CHUNK and lo - c1 < GAP:
            c1 = max(c1, fe)
            curr.append((i, lo, cs))
        else:
            chunks.append((c0, c1, curr))
            c0, c1, curr = lo, fe, [(i, lo, cs)]
    if curr:
        chunks.append((c0, c1, curr))
    print(f'  {len(file_info):,} files → {len(chunks):,} batch requests')
    failed = 0
    for c0, c1, c_files in tqdm(chunks, desc=f'{split}'):
        try:
            chunk = range_get(CALVIN_ZIP_URL, c0, c1 - 1, timeout=300)
        except Exception as e:
            print(f'\n  ! Chunk @ {c0}: {e}')
            failed += len(c_files)
            continue
        for i, lo, cs in c_files:
            try:
                rel  = lo - c0
                lhdr = chunk[rel:rel+30]
                assert lhdr[:4] == b'\x50\x4b\x03\x04'
                meth = struct.unpack_from('<H', lhdr,  8)[0]
                nl   = struct.unpack_from('<H', lhdr, 26)[0]
                el   = struct.unpack_from('<H', lhdr, 28)[0]
                doff = rel + 30 + nl + el
                raw  = chunk[doff:doff + cs]
                data = zlib.decompress(raw, -15) if meth == 8 else raw
                with open(f'{local_dir}/episode_{i:07d}.npz', 'wb') as fh:
                    fh.write(data)
            except Exception as e:
                failed += 1
                if failed <= 5:
                    print(f'\n  ! episode_{i:07d}: {e}')
    if failed:
        print(f'  WARNING: {failed}/{len(file_info)} files failed')

# ── Ensure lang_annotations on local disk ─────────────────────────────────────
def ensure_lang_ann(split):
    local = f'{CALVIN_PATH}/{split}/lang_annotations/auto_lang_ann.npy'
    if not os.path.exists(local):
        print(f'  [{split}] Downloading lang_annotations...')
        data = download_one(f'{split}/lang_annotations/auto_lang_ann.npy')
        with open(local, 'wb') as f:
            f.write(data)
        print(f'    {len(data)/1e6:.2f} MB')
    ann = np.load(local, allow_pickle=True).item()
    return ann['info']['indx'], ann['language']['task']

# ── Ensure episode NPZs (uses single glob, not per-file exists check) ─────────
def ensure_episodes(split, indx, max_ep):
    local_dir = f'{CALVIN_PATH}/{split}'
    needed    = sorted({i for s, e in indx[:max_ep] for i in range(int(s), int(e)+1)})
    # Single glob call — no per-file os.path.exists (that would take hours on Drive)
    on_disk   = {int(p.stem.replace('episode_', ''))
                 for p in Path(local_dir).glob('episode_*.npz')}
    missing   = [i for i in needed if i not in on_disk]
    print(f'  [{split}] {len(needed):,} needed | {len(on_disk):,} on disk | '
          f'{len(missing):,} to download')
    if missing:
        batch_download(split, missing, local_dir)
    final = len(list(Path(local_dir).glob('episode_*.npz')))
    print(f'  [{split}] {final:,} files on local disk')

# ── Tar helpers ───────────────────────────────────────────────────────────────
def create_and_save_tar():
    print('\n=== Step 3: Pack tar → Drive (done once, ~15 min total) ===')
    all_files = [p for p in Path(CALVIN_PATH).rglob('*') if p.is_file()]
    print(f'  Packing {len(all_files):,} files on local SSD...')
    with tarfile.open(TAR_LOCAL, 'w') as tf:
        for p in tqdm(all_files, desc='pack'):
            tf.add(str(p), arcname=str(p.relative_to(Path(CALVIN_PATH).parent)))
    sz = os.path.getsize(TAR_LOCAL) / 1e9
    print(f'  Tar: {sz:.2f} GB — copying to Drive (single file)...')
    shutil.copy2(TAR_LOCAL, TAR_DRIVE)
    os.remove(TAR_LOCAL)
    print(f'  Saved: {TAR_DRIVE}')

def restore_from_tar():
    print(f'Restoring from Drive tar (~3 min)...')
    base = Path(CALVIN_PATH).parent   # /content/calvin_data
    base.mkdir(parents=True, exist_ok=True)
    with tarfile.open(TAR_DRIVE, 'r') as tf:
        tf.extractall(str(base))
    n_tr = len(list(Path(f'{CALVIN_PATH}/training').glob('episode_*.npz')))
    n_va = len(list(Path(f'{CALVIN_PATH}/validation').glob('episode_*.npz')))
    print(f'  Extracted: training={n_tr:,}  validation={n_va:,}')

# ── Main setup ────────────────────────────────────────────────────────────────
lang_ok  = (os.path.exists(f'{CALVIN_PATH}/training/lang_annotations/auto_lang_ann.npy') and
            os.path.exists(f'{CALVIN_PATH}/validation/lang_annotations/auto_lang_ann.npy'))
train_ok = lang_ok and len(list(Path(f'{CALVIN_PATH}/training').glob('episode_*.npz'))) > 100

if train_ok:
    n_tr = len(list(Path(f'{CALVIN_PATH}/training').glob('episode_*.npz')))
    n_va = len(list(Path(f'{CALVIN_PATH}/validation').glob('episode_*.npz')))
    print(f'Data already on local disk: training={n_tr:,}  validation={n_va:,}')
elif os.path.exists(TAR_DRIVE):
    restore_from_tar()
else:
    print('=== Step 1: lang_annotations ===')
    train_indx, _ = ensure_lang_ann('training')
    val_indx,   _ = ensure_lang_ann('validation')
    print(f'  training: {len(train_indx):,}  validation: {len(val_indx):,}')
    print('\n=== Step 2: episode NPZs (batch range requests) ===')
    ensure_episodes('training',   train_indx, TRAIN_EP)
    ensure_episodes('validation', val_indx,   VAL_EP)
    create_and_save_tar()

# ── Verify ────────────────────────────────────────────────────────────────────
n_tr = len(list(Path(f'{CALVIN_PATH}/training').glob('episode_*.npz')))
n_va = len(list(Path(f'{CALVIN_PATH}/validation').glob('episode_*.npz')))
gb   = sum(p.stat().st_size for p in Path(CALVIN_PATH).rglob('*.npz')) / 1e9
print(f'\nVerify: training={n_tr:,}  validation={n_va:,}  ({gb:.1f} GB total)')
assert n_tr > 100, f'Too few training NPZs: {n_tr} — check errors above'
print(f'CALVIN_PATH = {CALVIN_PATH}')
print('Done ✓')


In [ ]:
# ── Cell 6: Smoke test — verify loader works before full run ─────────────────
import sys
sys.path.insert(0, REPO_DIR)

from data.trajectory_dataset import load_calvin

print('Loading a few CALVIN episodes to verify loader...')
eps = load_calvin(CALVIN_PATH, split='training')

print(f'Episodes loaded: {len(eps)}')
print(f'Keys: {list(eps[0].keys())}')
print(f'Frames shape:  {eps[0]["frames"].shape}')
print(f'Actions shape: {eps[0]["actions"].shape}  dtype={eps[0]["actions"].dtype}')
print(f'Action range:  {eps[0]["actions"].min()} – {eps[0]["actions"].max()}  (expect 0–13)')
print(f'Action vectors:{eps[0]["action_vectors"].shape}  (expect T×7)')
print(f'Instruction:   {eps[0]["instruction"]}')
print('Loader OK')

In [ ]:
# ── Cell 7: Configure run ─────────────────────────────────────────────────────
# ↓ ONLY change these two values ↓

START_FROM = 0    # Resume from ablation index if session died (0 = start fresh)
                  # Indices: 0=full_vera  1=bc_baseline  2=no_exp  3=no_act
                  #          4=no_lang    5=no_history_tf 6=no_reward_gate
                  #          7=no_dual_head  8=corrupted_conseq

DRY_RUN = False   # Set True first to verify (2 epochs, synthetic data, ~2 min)
                  # Set False for the real run

# ─────────────────────────────────────────────────────────────────────────────
# Point checkpoints to Drive so they survive session crashes
import yaml, os

cfg_path = f'{REPO_DIR}/configs/calvin_config.yaml'
with open(cfg_path) as f:
    cfg = yaml.safe_load(f)

# Redirect output to Drive
cfg['data']['episodes_path']   = CALVIN_PATH
cfg['training']['output_dir']  = f'{DRIVE_ROOT}/checkpoints'
cfg['training']['device']      = 'cuda'
cfg['training']['num_workers'] = 2    # Colab works better with 2

# Save patched config
colab_cfg_path = f'{REPO_DIR}/configs/calvin_colab_runtime.yaml'
with open(colab_cfg_path, 'w') as f:
    yaml.dump(cfg, f)

print('Config ready')
print(f'  Dataset:  {CALVIN_PATH}')
print(f'  Output:   {DRIVE_ROOT}/checkpoints')
print(f'  Epochs:   {cfg["training"]["epochs"]} (max, early stopping patience=10)')
print(f'  Ablations to run: {9 - START_FROM} (starting from index {START_FROM})')
print(f'  Seeds: 42, 123, 456')
print(f'  DRY RUN: {DRY_RUN}')

In [ ]:
# ── Cell 8: Run ablations ─────────────────────────────────────────────────────
# This cell runs all 9 ablations × 3 seeds sequentially.
# Checkpoints save to Drive after every epoch — safe to interrupt and resume.
# Expected time on T4:  ~20-40 min per (ablation × seed)  →  ~9-18 hours total
# Expected time on A100: ~8-15 min per (ablation × seed)  →  ~4-7  hours total

import subprocess, sys

cmd = [
    sys.executable,
    f'{REPO_DIR}/scripts/run_calvin_ablations.py',
    '--calvin_path', CALVIN_PATH,
    '--config',      colab_cfg_path,
    '--out',         f'{DRIVE_ROOT}/checkpoints',
    '--start_from',  str(START_FROM),
]
if DRY_RUN:
    cmd.append('--dry_run')

print('Running:', ' '.join(cmd))
print('='*65)

# Run with live output
result = subprocess.run(cmd, cwd=REPO_DIR)
print('='*65)
print('Exit code:', result.returncode)

In [ ]:
# ── Cell 9: View results ──────────────────────────────────────────────────────
import json, numpy as np

results_path = f'{DRIVE_ROOT}/checkpoints/calvin_ablation_summary.json'

with open(results_path) as f:
    results = json.load(f)

print(f'  {"Ablation":<46} {"Mean":>7}  {"Std":>6}  Seeds')
print(f'  {"-"*46} {"-"*7}  {"-"*6}  {"-"*22}')
for slug, v in results.items():
    seeds = '  '.join(f'{x:.4f}' for x in v['seeds'].values())
    print(f'  {v["display"]:<46} {v["mean_val_acc"]:.4f}  {v["std_val_acc"]:.4f}  {seeds}')

In [ ]:
# ── Cell 10: Download results JSON to local machine ───────────────────────────
from google.colab import files
import shutil, os

# Bundle the summary JSON + all training logs into one zip
zip_path = '/content/calvin_results.zip'
shutil.make_archive(
    '/content/calvin_results', 'zip',
    f'{DRIVE_ROOT}/checkpoints'
)
files.download(zip_path)
print('Download started')